# EdAcc — second spontaneous corpus (replication test)

Tests whether *spontaneous* speech defeats audio-free GPT-2 rescoring on an
**independent** corpus (EdAcc: Edinburgh International Accents of English), the
way it did on Svarah. EdAcc is public (CC-BY-SA, no token). **Runtime -> T4 GPU.**
You need `asr-l2-accent-code.zip` (from `make_upload_zips.py`).

In [ ]:
!nvidia-smi -L

In [ ]:
!pip install -q datasets faster-whisper transformers jiwer librosa soundfile pandas pyarrow pyyaml tqdm

In [ ]:
import zipfile, os
from google.colab import files
up = files.upload()                      # asr-l2-accent-code.zip
with zipfile.ZipFile(next(iter(up))) as z:
    z.extractall('/content')
os.chdir('/content/asr-l2-accent'); print('cwd:', os.getcwd())

## 1. Download EdAcc (subsample, ~200 spontaneous clips)
The schema is auto-detected and printed. If the text/accent column mapping looks
wrong, re-run with the right names patched into `download_edacc.py`.

In [ ]:
!python scripts/download_edacc.py --split validation --max-utterances 200

## 2. Whisper n-best + GPT-2 rescoring on EdAcc
Same frozen Whisper-small.en, same audio-free GPT-2 rescoring as the main study.
The question: does GPT-2 help on spontaneous EdAcc (expected: little/none, as on Svarah)?

In [ ]:
!python scripts/rescore_nbest.py --dataset edacc --lm neural:gpt2 --max-utterances 200

In [ ]:
!python scripts/bootstrap_significance.py --dataset edacc

In [ ]:
import json
s = json.load(open("results/edacc/phaseB_summary.json"))
b = s["test_wer_baseline"]
print(f"EdAcc GPT-2 rescoring: WER {b:.4f} -> {s['test_wer_adapted']:.4f} "
      f"({100*s['test_wer_delta']/b:+.1f}%)  lambda*={s['best_lambda']}")
print("=> spontaneous-speech null replicates" if s['test_wer_delta'] >= -0.001
      else "=> gain on EdAcc (differs from Svarah) -- worth a closer look")

## 3. Download results

In [ ]:
import zipfile, glob
with zipfile.ZipFile("phaseB_edacc_results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob("results/edacc/*.json") + glob.glob("results/edacc/*.csv") + \
             glob.glob("data/processed/edacc/integrity.json"):
        z.write(f)
from google.colab import files
files.download("phaseB_edacc_results.zip")